# Assignment 3 CPSC 542

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import models
from torchvision.transforms import v2
from tqdm import tqdm
import kagglehub
from torchvision.io import read_image
from torchvision.tv_tensors import BoundingBoxes
import cv2
from ensemble_boxes import weighted_boxes_fusion



/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
print(np.__version__)

2.2.6


In [23]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("awsaf49/vinbigdata-1024-image-dataset")

print("Path to dataset files:", path)

Resuming download from 274726912 bytes (8466603624 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/1.archive (274726912/8741330536) bytes left.


100%|██████████| 8.14G/8.14G [02:13<00:00, 63.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1


* Note: We don't want to use the test set here because that is part of the competition scoring, we will make our own train/val/test splits using the testing set

In [2]:
records = pd.read_csv("vinbigdata-1024-image-dataset/versions/1/vinbigdata/train.csv")


In [3]:
records.head()


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,width,height
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN,2332,2580
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN,2954,3159
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0,2080,2336
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0,2304,2880
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN,2540,3072


In [4]:
records.isna().sum()

image_id          0
class_name        0
class_id          0
rad_id            0
x_min         31818
y_min         31818
x_max         31818
y_max         31818
width             0
height            0
dtype: int64

## EDA and Pipeline for New Dataset

Handle Missing or Impossible Values

In [5]:
records.shape


(67914, 10)

### Consolidate Images based on rad_id

We have 67,914 rows in the training set because mutliple radiologists have annotated each image, rad_id tells us which radiologist did the annotation, so we will use Weighted Box Fusion to combine xrays with annotations from different radiologists on the same x-ray into one solid set.

In [6]:
def consolidate_dataset(df, iou_thr=0.5, skip_box_thr=0.0001):
    # Separate 'No Finding' (Class 14) 
    findings_df = df[df['class_id'] != 14].copy()
    
    # Pre-calculate mapping for class_id -> class_name to keep it fast
    class_map = findings_df[['class_id', 'class_name']].drop_duplicates().set_index('class_id')['class_name'].to_dict()
    # Pre-calculate mapping for image_id -> (width, height)
    dim_map = df[['image_id', 'width', 'height']].drop_duplicates().set_index('image_id').to_dict('index')
    
    # Normalize coordinates for WBF [0, 1]
    findings_df['x_min'] /= findings_df['width']
    findings_df['y_min'] /= findings_df['height']
    findings_df['x_max'] /= findings_df['width']
    findings_df['y_max'] /= findings_df['height']
    
    new_rows = []
    image_ids = findings_df['image_id'].unique()
    
    for img_id in tqdm(image_ids, desc="Fusing Boxes"):
        img_group = findings_df[findings_df['image_id'] == img_id]
        
        boxes_list = [img_group[['x_min', 'y_min', 'x_max', 'y_max']].values.tolist()]
        scores_list = [[1.0] * len(img_group)] 
        labels_list = [img_group['class_id'].values.tolist()]
        
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list, 
            weights=None, iou_thr=iou_thr, skip_box_thr=skip_box_thr
        )
        
        # Get dimensions once for this image
        w = dim_map[img_id]['width']
        h = dim_map[img_id]['height']
        
        for i in range(len(boxes)):
            cls_id = int(labels[i])
            new_rows.append({
                'image_id': img_id,
                'class_id': cls_id,
                'class_name': class_map[cls_id],
                'width': w,
                'height': h,
                'x_min': boxes[i][0],
                'y_min': boxes[i][1],
                'x_max': boxes[i][2],
                'y_max': boxes[i][3]
            })
            
    return pd.DataFrame(new_rows)

consolidated_records = consolidate_dataset(records)

Fusing Boxes:   0%|          | 0/4394 [00:00<?, ?it/s]

Fusing Boxes: 100%|██████████| 4394/4394 [00:15<00:00, 290.19it/s]


Re-introduce No Finding and change the NaN bounding boxes to "0 0 1 1" as specified on the kaggle site. We want No Finding to have a confidence of 1 with that 1 pixel bounding box.

In [7]:

no_findings_unique = records[records['class_id'] == 14].drop_duplicates(subset='image_id')

# Clean up the coordinates for No Finding (using the Kaggle 1x1 pixel convention)
no_findings_unique[['x_min', 'y_min', 'x_max', 'y_max']] = [0, 0, 1, 1] 

# Combine them
final_records = pd.concat([consolidated_records, no_findings_unique], ignore_index=True)
final_records = final_records.drop(columns=['rad_id'])

In [8]:
final_records.head()

,image_id,class_id,class_name,width,height,x_min,y_min,x_max,y_max
0,9a5094b2563a1ef3ff50dc5c7ff71345,0,Aortic enlargement,2080,2336,0.505769,0.306079,0.624519,0.413527
1,9a5094b2563a1ef3ff50dc5c7ff71345,11,Pleural thickening,2080,2336,0.860096,0.740154,0.901442,0.852740
2,9a5094b2563a1ef3ff50dc5c7ff71345,10,Pleural effusion,2080,2336,0.860096,0.740154,0.901442,0.852740
3,9a5094b2563a1ef3ff50dc5c7ff71345,3,Cardiomegaly,2080,2336,0.332051,0.579766,0.797436,0.769549
4,051132a778e61a86eb147c7c6f564dfe,11,Pleural thickening,2304,2880,0.690972,0.156944,0.782986,0.209722


In [9]:
final_records.shape

(34540, 9)

In [10]:
final_records.isna().sum()

image_id      0
class_id      0
class_name    0
width         0
height        0
x_min         0
y_min         0
x_max         0
y_max         0
dtype: int64

Now we have a cleaned dataset with "combined" radiologist bounding boxes, no missing values.

Check for how many unique images there are since we have repeats

In [11]:
print("Total number of patients: ", records['image_id'].nunique(), "\n")

Total number of patients:  15000 



We have 15,000 unique images

### Data Augmentation

In [24]:
import os

# 1. The ID that failed
problem_id = "d3ef9ae515fb3cd3565e2c9875b3e0aa"
IMAGE_DIRECTORY = "/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train"

# 2. Check for hidden characters in the ID (like spaces)
print(f"Original ID length: {len(problem_id)}")
clean_id = problem_id.strip()
print(f"Cleaned ID length: {len(clean_id)}")

# 3. Construct the path manually
constructed_path = os.path.join(IMAGE_DIRECTORY, f"{clean_id}.png")
print(f"Full path we are testing: '{constructed_path}'")

# 4. Check if that specific file exists
exists = os.path.isfile(constructed_path)
print(f"File exists at that exact path: {exists}")

# 5. List the first few actual filenames on disk to check extensions/case
actual_files = os.listdir(IMAGE_DIRECTORY)
print(f"First 5 actual files on disk: {actual_files[:5]}")

Original ID length: 32
Cleaned ID length: 32
Full path we are testing: '/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train/d3ef9ae515fb3cd3565e2c9875b3e0aa.png'
File exists at that exact path: True
First 5 actual files on disk: ['a04cd3322f452395867fa98d676e9486.png', 'd4d41e0e0e7c512d05cdbd89401ee7bf.png', 'f3185be657c553fac1498f0d8088d56d.png', '0f34355eca9e6fb0fabab21711f0a2ec.png', 'd0252f72f9db197d04d2f59c8f77f908.png']


In [25]:
train_transforms = v2.Compose([
    v2.ToImage(),                          # Convert to tensor-friendly format
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=10),
    v2.ColorJitter(brightness=0.2, contrast=0.2), 
    v2.Resize(size=(1024, 1024), antialias=True),
    v2.ToDtype(torch.float32, scale=True), # Scales 0-255 to 0-1
    v2.Normalize(mean=[0.485], std=[0.229])
])

IMAGE_DIRECTORY = "/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train"

class VinBigDataDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        self.img_ids = df['image_id'].unique()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = f"{self.img_dir}/{img_id}.png"
        
        img = read_image(img_path) 
        
        # Get Boxes
        final_records = self.df[self.df['image_id'] == img_id]
        bboxes = final_records[['x_min', 'y_min', 'x_max', 'y_max']].values
        labels = torch.tensor(final_records['class_id'].values, dtype=torch.int64)

        # Wrap boxes in Torchvision's TVTensor to enable auto-augmentation
        bboxes = BoundingBoxes(bboxes, format="XYXY", canvas_size=img.shape[-2:])

        if self.transforms:
            img, bboxes, labels = self.transforms(img, bboxes, labels)

        target = {
            "boxes": bboxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return img, target

In [26]:
val_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize(size=(1024, 1024), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485], std=[0.229])
])

### Train, Validation and Test Splits

In [27]:
files_on_disk = [f.replace('.png', '') for f in os.listdir(IMAGE_DIRECTORY) if f.endswith('.png')]

# 2. Filter your main dataframe to only include these images
final_records_filtered = final_records[final_records['image_id'].isin(files_on_disk)].copy()

print(f"Original rows: {len(final_records)}")
print(f"Rows after filtering for existing images: {len(final_records_filtered)}")
print(f"Unique images available: {final_records_filtered['image_id'].nunique()}")

Original rows: 34540
Rows after filtering for existing images: 34540
Unique images available: 15000


In [28]:
# Want to split based on unique images
unique_images = final_records['image_id'].unique()

train_val_ids, test_ids = train_test_split(unique_images, test_size=0.10, random_state=42)

train_ids, val_ids = train_test_split(train_val_ids, test_size=0.11, random_state=42)

print(f"Train images: {len(train_ids)} | Val images: {len(val_ids)} | Test images: {len(test_ids)}")

Train images: 12015 | Val images: 1485 | Test images: 1500


Apply Transforms to the splits

In [29]:
train_df = final_records[final_records['image_id'].isin(train_ids)]
val_df = final_records[final_records['image_id'].isin(val_ids)]
test_df = final_records[final_records['image_id'].isin(test_ids)]


Create teh datasets with mappings and transforms

In [30]:
# Training Dataset
train_dataset = VinBigDataDataset(
    df=train_df, 
    img_dir=IMAGE_DIRECTORY,   # <--- The Mapping Happens Here
    transforms=train_transforms
)

# Validation Dataset
val_dataset = VinBigDataDataset(
    df=val_df, 
    img_dir=IMAGE_DIRECTORY, 
    transforms=val_transforms
)

# Test Dataset
test_dataset = VinBigDataDataset(
    df=test_df, 
    img_dir=IMAGE_DIRECTORY, 
    transforms=val_transforms
)

### DataLoader

In [31]:
def collate_fn(batch):
    """
    Handles variable numbers of bounding boxes per image in a batch.
    """
    return tuple(zip(*batch))

In [ ]:

# Hopefully 4 is fine for batch size 
BATCH_SIZE = 4 

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True  
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True
)

Test Pipeline

In [ ]:
images, targets = next(iter(train_loader))

print(f"--- Data Pipeline Verification ---")
print(f"Batch Size: {len(images)}")
print(f"Image Tensor Shape: {images[0].shape}") # Should be [3, 1024, 1024]
print(f"Number of Boxes in first image: {len(targets[0]['boxes'])}")
print(f"Labels in first image: {targets[0]['labels']}")
print(f"Device: {'GPU Ready' if images[0].is_cuda == False else 'Already on GPU'}")

--- Data Pipeline Verification ---
Batch Size: 4
Image Tensor Shape: torch.Size([1, 1024, 1024])
Number of Boxes in first image: 3
Labels in first image: tensor([13, 13,  6])
Device: GPU Ready
